# Агент, инструменты и оптимизация harness

**За 25–35 минут:** пройти один агентный цикл, прочитать траекторию, изменить конфигурацию и проверить гипотезу.
В примере считаем выручку по статусу и валюте на маленькой таблице заказов в SQLite.
Нужны базовые Python и SQL; запустите все ячейки по порядку (Run All).
Откройте `notebooks/01_agent_and_harness.ipynb` внутри клона этого репозитория; из его корня установите `python -m pip install -e ".[notebooks]"` в `.venv` и выберите этот kernel.

Все ячейки по умолчанию работают **локально, без API, PostgreSQL и Docker**.
Используем настоящий `HarnessAgent`, `HarnessConfig` и `ToolExecutor` проекта; вместо LLM — прозрачная программная `ScriptedPolicy`.
SQL и проверки здесь исполняются; поведение policy задано нами. Это учебная демонстрация механики, а не измерение качества LLM.


## 1. Что делает агент

`цель → вызов модели → tool calls → выполнение Python-кода → observation → следующий вызов → ответ/лимит`

Модель предлагает действие. Harness добавляет его результат в историю и решает, продолжать ли цикл.
JSON-схема инструмента объясняет, **какие аргументы запросить**; исполняющий код определяет, **что реально произойдёт**.
Мета-оптимизатор находится снаружи: предлагает patch конфигурации и сравнивает новые прогоны. Веса модели он не обучает.


In [2]:
import json, re, sys
from pathlib import Path
from statistics import mean
from types import SimpleNamespace

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/harness_lab/agent.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Откройте notebook в клоне workshop repository; нужен src/harness_lab. См. README рядом.")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "notebooks"))
from harness_lab.agent import HarnessAgent
from harness_lab.cases import BenchmarkCase, DialogueTurn
from harness_lab.config import HarnessConfig, SearchConfig
from harness_lab.model import ModelTurn, ToolCall
from harness_lab.tools import ToolExecutor, build_tool_schemas
from harness_lab.optimizer import MetaHarnessOptimizer, MUTABLE_FIELDS
from harness_lab.runner import config_fingerprint
from _support import SQLiteSandbox
print("Импортирован основной агент проекта. Сеть и .env не используются.")


Импортирован основной агент проекта. Сеть и .env не используются.


## 2. Прикладная задача: сводка оплаченной выручки

Нужно вернуть контракт отчёта и `observed_value`: сумму `amount_cents` только для заданных `status` и `currency`.
Суммы храним в целых минимальных единицах валюты, разные валюты не складываем.
В каждом прогоне среда получает только исходные строки; ожидаемое значение вычисляет отдельная проверка после ответа.
Здесь учебная `report`-задача требует чтения SQLite, поэтому `routing='none'`; в основном benchmark `report` собирает ТЗ из диалога без БД.
Минимальная метрика — точное совпадение суммы. Она **не проверяет весь контракт отчёта**, корректность произвольного SQL или продакшен-безопасность.


In [3]:
ROWS = [
    (1, "paid", "RUB", 12000), (2, "paid", "RUB", 3500),
    (3, "paid", "USD", 900), (4, "cancelled", "RUB", 50000),
    (5, "refunded", "USD", 400), (6, "refunded", "RUB", 1800),
]
def task(case_id, status, currency, rows):
    params = {"status": status, "currency": currency}
    case = BenchmarkCase(id=case_id, kind="report", title="Выручка по статусу и валюте",
        dialogue=[DialogueTurn(role="user", content=(
            "Составь контракт отчета по заказам orders и посчитай сумму amount_cents. "
            "Применяй оба фильтра; добавь observed_value в метрику revenue. "
            "Инструменты этого учебного примера работают с SQLite.\n"
            "Параметры задачи: " + json.dumps(params)))])
    return {"case": case, "params": params, "rows": rows}

train = [task("train_rub", "paid", "RUB", ROWS), task("train_usd", "paid", "USD", ROWS)]
print(train[0]["case"].dialogue[0].content)
print("Исходные заказы:", ROWS)


Составь контракт отчета по заказам orders и посчитай сумму amount_cents. Применяй оба фильтра; добавь observed_value в метрику revenue. Инструменты этого учебного примера работают с SQLite.
Параметры задачи: {"status": "paid", "currency": "RUB"}
Исходные заказы: [(1, 'paid', 'RUB', 12000), (2, 'paid', 'RUB', 3500), (3, 'paid', 'USD', 900), (4, 'cancelled', 'RUB', 50000), (5, 'refunded', 'USD', 400), (6, 'refunded', 'RUB', 1800)]


## 3. Схема и исполнитель — разные части

Берём JSON-схемы из `src/harness_lab/tools.py`. Их описания ориентированы на PostgreSQL;
в этой локальной демонстрации только `describe_table` и `run_readonly_query` реализованы через адаптер SQLite.
Короткий адаптер доступен отдельно: [_support.py](_support.py).
Схема `query: string` сама не делает запрос безопасным: ограничения обязан применять исполнитель.
`ToolExecutor.execute()` разбирает аргументы, вызывает адаптер и возвращает JSON `ok/result` или `ok/error`.


In [4]:
baseline = HarnessConfig(name="baseline", enabled_tools=["run_readonly_query"], max_steps=3)
print(json.dumps(build_tool_schemas(baseline)[0], ensure_ascii=False, indent=2))
sandbox = SQLiteSandbox(ROWS)
try:
    executor = ToolExecutor(sandbox)
    print("Реальный локальный SELECT:", executor.execute("run_readonly_query", '{"query":"SELECT COUNT(*) AS n FROM orders"}'))
    print("Отклоненное действие:", executor.execute("run_readonly_query", '{"query":"DELETE FROM orders"}'))
finally:
    sandbox.close()


{
  "type": "function",
  "function": {
    "name": "run_readonly_query",
    "description": "Run one read-only SELECT query and return up to 50 rows.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string"
        }
      },
      "required": [
        "query"
      ],
      "additionalProperties": false
    }
  }
}
Реальный локальный SELECT: {"ok": true, "result": {"rows": [{"n": 6}], "truncated": false}}
Отклоненное действие: {"ok": false, "error": "ValueError", "message": "The teaching adapter accepts one SELECT only"}


In [5]:
def report(query, value):
    return {"report_name": "revenue_by_population", "objective": "Выручка выбранной популяции",
        "grain": "one requested status and currency", "timezone": "UTC",
        "population": {"include": ["requested status and currency"], "exclude": []},
        "metrics": [{"id": "revenue", "definition": "Сумма amount_cents", "formula_sql": query,
                     "owner": "finance", "observed_value": value}],
        "dimensions": [], "filters": [], "comparisons": [],
        "delivery": {"format": "JSON", "cadence": "on demand"},
        "assumptions": [], "open_questions": [], "validation_checks": ["exact amount check"]}

class ScriptedPolicy:
    '''Показывает интерфейс complete(); это Python-программа, а не LLM.'''
    def __init__(self):
        self.query = ""

    def complete(self, messages, tools=None, **kwargs):
        if "domain-specialist subagent" in messages[0]["content"]:
            # Учебный review возвращает тот же ответ, ничего не «оценивает».
            return ModelTurn(messages[-1]["content"].split("CANDIDATE\n", 1)[1])
        text = "\n".join(m.get("content") or "" for m in messages if m["role"] == "user")
        params = json.loads(re.search(r"Параметры задачи: (\{[^\n]+\})", text).group(1))
        if "Return a concise numbered plan" in text:
            return ModelTurn("1. Проверить схему. 2. Применить фильтры. 3. Выполнить SELECT.")
        observations = [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
        inspected = any("columns" in o.get("result", {}) for o in observations)
        aggregation = next((o["result"]["rows"] for o in observations if "rows" in o.get("result", {})), None)
        names = {t["function"]["name"] for t in (tools or [])}
        if "describe_table" in names and not observations:
            return ModelTurn("", [ToolCall("schema-1", "describe_table", '{"table":"orders"}')])
        if "run_readonly_query" in names and aggregation is None:
            # Заданная учебная ошибка: быстрый путь забывает валюту.
            self.query = "SELECT COALESCE(SUM(amount_cents), 0) AS amount FROM orders WHERE status='" + params["status"] + "'"
            if inspected:
                self.query += " AND currency='" + params["currency"] + "'"
            return ModelTurn("", [ToolCall("sum-1", "run_readonly_query", json.dumps({"query": self.query}))])
        value = aggregation[0]["amount"] if aggregation else None
        return ModelTurn(json.dumps(report(self.query, value), ensure_ascii=False))


`ScriptedPolicy` специально устроена так: если доступна проверка схемы, она сначала вызывает её,
а после этого учитывает оба фильтра; быстрый путь пропускает валюту. Эти ветви видны в коде.
Никакой настоящей модели мы не заставили улучшиться: этот контрпример нужен, чтобы отличить
изменение процесса от простого увеличения бюджета. В реальном эксперименте каждую гипотезу придётся измерить заново.

Ниже явно записан минимальный ReAct-цикл: найдите момент получения tool call, исполнения инструмента и добавления observation.
Он повторяет основу `HarnessAgent.run`; планирование, review и forced final затем посмотрим в настоящем harness.


In [6]:
from harness_lab.prompts import system_prompt, render_task
model = ScriptedPolicy()
db = SQLiteSandbox(train[0]["rows"])
executor = ToolExecutor(db)
messages = [{"role": "system", "content": system_prompt(train[0]["case"], baseline)},
            {"role": "user", "content": render_task(train[0]["case"])}]
visible_answer = None
try:
    for step in range(1, baseline.max_steps + 1):
        turn = model.complete(messages, tools=build_tool_schemas(baseline))
        messages.append(turn.as_message())
        if not turn.tool_calls:
            visible_answer = turn.content
            print("STOP: policy вернула ответ без tool calls")
            break
        for call in turn.tool_calls:
            observation = executor.execute(call.name, call.arguments)
            print("TOOL:", call.name, "OBSERVATION:", observation)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": observation})
finally:
    db.close()
print("Роли в истории:", [m["role"] for m in messages])


TOOL: run_readonly_query OBSERVATION: {"ok": true, "result": {"rows": [{"amount": 16400}], "truncated": false}}
STOP: policy вернула ответ без tool calls
Роли в истории: ['system', 'user', 'assistant', 'tool', 'assistant']


In [7]:
def episode(item, config):
    sandbox = SQLiteSandbox(item["rows"])
    try:
        result = HarnessAgent(ScriptedPolicy(), ToolExecutor(sandbox)).run(item["case"], config)
    finally:
        sandbox.close()
    answer = json.loads(result.output) if result.output else {}
    value = answer.get("metrics", [{}])[0].get("observed_value")
    p = item["params"]
    expected = sum(amount for _, status, currency, amount in item["rows"]
                   if status == p["status"] and currency == p["currency"])
    return {"task": item["case"].id, "passed": result.error is None and value == expected,
            "value": value, "expected": expected, "policy_calls": result.llm_calls,
            "tool_calls": sum(e.stage == "tool" for e in result.events), "result": result}

def show_trace(row):
    for e in row["result"].events:
        if e.stage == "tool":
            print(f"step {e.data['step']} → {e.data['name']}({e.data['arguments']})")
            print("  observation:", e.data["output"])
        else:
            print("harness:", e.stage, e.data)
    print("Проверка:", {k: v for k, v in row.items() if k != "result"})

first = episode(train[0], baseline)
assert visible_answer == first["result"].output
show_trace(first)


step 1 → run_readonly_query({"query": "SELECT COALESCE(SUM(amount_cents), 0) AS amount FROM orders WHERE status='paid'"})
  observation: {"ok": true, "result": {"rows": [{"amount": 16400}], "truncated": false}}
harness: react_step {'step': 1, 'tools': ['run_readonly_query']}
harness: final_candidate {'step': 2}
Проверка: {'task': 'train_rub', 'passed': False, 'value': 16400, 'expected': 15500, 'policy_calls': 2, 'tool_calls': 1}


## 4. Что меняем в harness

- `enabled_tools` и описания — что модель видит в схемах; исполнитель остаётся отдельным кодом.
- `prompt_patch`, `prompt_profile` — дополнительные инструкции. `routing='task_kind'` использует уже заданный `case.kind`: report-задачи получают **пустой список схем**. Это не вызов LLM-классификатора.
- `workflow='plan_execute'` — дополнительный вызов той же модели для плана перед основным циклом.
- `review_mode='domain_specialist'` — отдельный контекст и ещё один вызов той же модели без инструментов; возвращённый текст **заменяет** ответ. Это не независимый grader.
- `context_strategy='requirements_ledger'` — предварительное сжатие каждого report-диалога моделью, если выбран этот режим; порог заполнения окна не измеряется. `context_partition` меняет контекст review.
- `max_steps` ограничивает обороты цикла; planning, ledger, review и принудительный final могут добавлять вызовы сверх этого числа.

`max_context_chars` обрезает **начальный текст задачи в символах**, сохраняя начало и конец.
Далее история инструментов накапливается целиком: динамического token-reserve или маскирования observations здесь нет.
В свободном чате demo маршрут выбирает простая эвристика по ключевым словам последних четырёх сообщений;
его цикл отличается от benchmark и не применяет ledger, planning и review.

В этом коротком примере сравним две гипотезы: **A — больше шагов**, **B — сначала схема**.


In [8]:
proposals = [
    {"hypothesis": "Дополнительные шаги дадут возможность исправить ответ.", "patch": {"max_steps": 8}},
    {"hypothesis": "Проверка схемы перед SELECT поможет применить всю популяцию.",
     "patch": {"enabled_tools": ["describe_table", "run_readonly_query"]}},
]
def apply_patch(parent, proposal):
    patch = proposal["patch"]
    if not 1 <= len(patch) <= 4 or not set(patch) <= MUTABLE_FIELDS:
        raise ValueError("Нужны 1–4 разрешенных поля основного optimizer")
    candidate = HarnessConfig.model_validate({**parent.model_dump(), **patch})
    changed = MetaHarnessOptimizer._config_diff(parent, candidate)
    if len(changed) != len(patch):
        raise ValueError("Patch должен реально менять каждое указанное поле")
    return candidate, changed

more_steps, _ = apply_patch(baseline, proposals[0])
schema_first, diff = apply_patch(baseline, proposals[1])
print("Patch B:", json.dumps(diff, ensure_ascii=False, indent=2))
show_trace(episode(train[0], schema_first))


Patch B: {
  "enabled_tools": {
    "from": [
      "run_readonly_query"
    ],
    "to": [
      "describe_table",
      "run_readonly_query"
    ]
  }
}
step 1 → describe_table({"table":"orders"})
  observation: {"ok": true, "result": {"table": "orders", "columns": [{"cid": 0, "name": "id", "type": "INTEGER", "notnull": 0, "dflt_value": null, "pk": 0}, {"cid": 1, "name": "status", "type": "TEXT", "notnull": 0, "dflt_value": null, "pk": 0}, {"cid": 2, "name": "currency", "type": "TEXT", "notnull": 0, "dflt_value": null, "pk": 0}, {"cid": 3, "name": "amount_cents", "type": "INTEGER", "notnull": 0, "dflt_value": null, "pk": 0}]}}
harness: react_step {'step': 1, 'tools': ['describe_table']}
step 2 → run_readonly_query({"query": "SELECT COALESCE(SUM(amount_cents), 0) AS amount FROM orders WHERE status='paid' AND currency='RUB'"})
  observation: {"ok": true, "result": {"rows": [{"amount": 15500}], "truncated": false}}
harness: react_step {'step': 2, 'tools': ['run_readonly_query']}
harness

## 5. Метацикл: гипотеза → patch → прогон → выбор

Считаем долю точных ответов и количество вызовов. Для учебного выбора используем ту же формулу
`MetaHarnessOptimizer.objective`: `score − 0.006 × max(avg_calls − 1, 0)`.
Штраф задаёт компромисс, а не универсальную «цену» качества. Токены и деньги здесь не измеряем: настоящей LLM нет.

Ниже два **локальных варианта генерации гипотез**: заранее записанный набор и Python-правило по train-траектории.
В реальном `MetaHarnessOptimizer` вместо них вызывается модель, а JSON-предложения проходят более подробную проверку разнообразия и смысла изменений.


In [9]:
def evaluate(config, items):
    rows = [episode(item, config) for item in items]
    summary = {"score": mean(r["passed"] for r in rows),
               "avg_llm_calls": mean(r["policy_calls"] for r in rows)}
    holder = SimpleNamespace(search=SearchConfig(objective_call_penalty=.006))
    objective = MetaHarnessOptimizer.objective(holder, SimpleNamespace(summary=summary))
    return {"config": config, "rows": rows, **summary, "objective": objective}

def table(states):
    print("config | exact accuracy | policy calls | objective")
    for s in states:
        print(f"{s['config'].name}: {s['score']:.0%} | {s['avg_llm_calls']:.1f} | {s['objective']:.3f}")

comparison = [evaluate(c.model_copy(update={"name": n}), train)
              for n, c in [("baseline", baseline), ("A_more_steps", more_steps), ("B_schema_first", schema_first)]]
table(comparison)


config | exact accuracy | policy calls | objective
baseline: 0% | 2.0 | -0.006
A_more_steps: 0% | 2.0 | -0.006
B_schema_first: 100% | 3.0 | 0.988


In [10]:
def predefined_proposer(train_feedback):
    return proposals  # Учебные заранее записанные гипотезы, не вывод LLM.

def trace_rule_proposer(train_feedback):
    failed = [r for r in train_feedback if not r["passed"]]
    missing_currency = any("currency" not in r["query"] for r in failed)
    return [proposals[1]] if missing_currency else [proposals[0]]

def search_on_train(proposer):
    initial = evaluate(baseline, train)
    feedback = [{"task": r["task"], "passed": r["passed"],
                 "query": json.loads(r["result"].output)["metrics"][0]["formula_sql"]}
                for r in initial["rows"]]
    states, history = [initial], []
    for i, proposal in enumerate(proposer(feedback), 1):
        cfg, changes = apply_patch(baseline, proposal)
        cfg = cfg.model_copy(update={"name": f"candidate_{i}"})
        state = evaluate(cfg, train)
        states.append(state)
        history.append({"hypothesis": proposal["hypothesis"], "diff": changes,
                        "train_score": state["score"], "objective": state["objective"]})
    return {"states": states, "train_winner": max(states, key=lambda s: s["objective"]), "history": history}

search_a, search_b = search_on_train(predefined_proposer), search_on_train(trace_rule_proposer)
print("Вариант A: фиксированные гипотезы"); table(search_a["states"])
print("Вариант B: правило по train trace"); table(search_b["states"])
print("История A:", json.dumps(search_a["history"], ensure_ascii=False, indent=2))


Вариант A: фиксированные гипотезы
config | exact accuracy | policy calls | objective
baseline: 0% | 2.0 | -0.006
candidate_1: 0% | 2.0 | -0.006
candidate_2: 100% | 3.0 | 0.988
Вариант B: правило по train trace
config | exact accuracy | policy calls | objective
baseline: 0% | 2.0 | -0.006
candidate_1: 100% | 3.0 | 0.988
История A: [
  {
    "hypothesis": "Дополнительные шаги дадут возможность исправить ответ.",
    "diff": {
      "max_steps": {
        "from": 3,
        "to": 8
      }
    },
    "train_score": 0,
    "objective": -0.006
  },
  {
    "hypothesis": "Проверка схемы перед SELECT поможет применить всю популяцию.",
    "diff": {
      "enabled_tools": {
        "from": [
          "run_readonly_query"
        ],
        "to": [
          "describe_table",
          "run_readonly_query"
        ]
      }
    },
    "train_score": 1,
    "objective": 0.988
  }
]


## 6. Где заканчивается поиск и начинается оценка

**Основной workshop сейчас:** 10 train (6 report, 4 SQL) и 7 открытых eval (4 report, 3 SQL).
Его `eval` можно изучать и повторять; это **не закрытый test**. Исторические результаты не переименовываем в holdout.

**Рекомендуемый следующий эксперимент:** train — предложения и их проверка; dev — выбор среди уже полученных конфигураций;
затем фиксируем победителя и один раз открываем test. Если дорабатываем после test, нужен новый test.
Ниже отдельная маленькая учебная демонстрация такого разделения. Четыре дополнительных сценария не дают статистического доказательства переноса.


In [11]:
dev = [task("dev_refunds_rub", "refunded", "RUB", ROWS), task("dev_cancelled_usd", "cancelled", "USD", ROWS)]
fresh_rows = ROWS + [(7, "paid", "USD", 2100), (8, "paid", "EUR", 1700)]
test = [task("test_new_usd", "paid", "USD", fresh_rows), task("test_new_eur", "paid", "EUR", fresh_rows)]

# Генерация предложений уже закончена. Dev не передается proposer.
dev_states = [evaluate(s["config"], dev) for s in search_a["states"]]
selected = max(dev_states, key=lambda s: s["objective"])["config"]
frozen_fingerprint = config_fingerprint(selected)
print("Выбор на dev:"); table(dev_states)
print("Frozen config fingerprint:", frozen_fingerprint)
test_result = evaluate(selected, test)
assert config_fingerprint(selected) == frozen_fingerprint
print("Одна итоговая проверка на test:"); table([test_result])


Выбор на dev:
config | exact accuracy | policy calls | objective
baseline: 0% | 2.0 | -0.006
candidate_1: 0% | 2.0 | -0.006
candidate_2: 100% | 3.0 | 0.988
Frozen config fingerprint: 986d21dc8b
Одна итоговая проверка на test:
config | exact accuracy | policy calls | objective
candidate_2: 100% | 3.0 | 0.988


## 7. Почему лимит шагов не равен числу вызовов

Следующие три прогона показывают настоящее поведение основного harness с нашей policy:
план добавляет вызов перед циклом; review добавляет вызов после него; при слишком коротком цикле
`forced_final` делает ещё один вызов модели без инструментов и просит итоговый JSON.
Это не извлечение JSON из готового текста и не гарантия выполнения задачи; parser grader работает отдельно.
Наш scripted review просто возвращает кандидата без изменений — качество review этим примером не измеряется.

В основном SQL benchmark инструкции про схему, EXPLAIN и пробный индекс подталкивают модель к проверкам;
цикл не блокирует ответ, пока эти действия не выполнены. Отдельный grader проверяет траекторию,
результат SELECT и измерения плана после ответа. Наша локальная метрика суммы эти SQL-проверки не заменяет.


In [12]:
for label, patch in [("plan_execute", {"workflow": "plan_execute"}),
                     ("review", {"review_mode": "domain_specialist"}),
                     ("too_short", {"max_steps": 1})]:
    row = episode(train[0], schema_first.model_copy(update=patch))
    print(label, "policy calls:", row["policy_calls"], "passed:", row["passed"],
          "stages:", [e.stage for e in row["result"].events])
assert first["passed"] is False
assert comparison[1]["score"] == 0  # Увеличение бюджета не изменило policy.
assert test_result["score"] == 1
print("Локальные проверки выполнены. Это не измерение качества или стоимости LLM.")


plan_execute policy calls: 4 passed: True stages: ['planning', 'tool', 'react_step', 'tool', 'react_step', 'final_candidate']
review policy calls: 4 passed: True stages: ['tool', 'react_step', 'tool', 'react_step', 'final_candidate', 'domain_specialist_review']
too_short policy calls: 2 passed: False stages: ['tool', 'react_step', 'forced_final']
Локальные проверки выполнены. Это не измерение качества или стоимости LLM.


## 8. Настоящая модель — необязательный следующий шаг

В обеих live-страницах есть панель подключения: URL, ключ, модель, прокси, CA и проверка `/models`.
Её профили живут в памяти **HTTP-сервера**, автоматически в kernel ноутбука не передаются.
Для ноутбука задайте постоянное подключение в корневом `.env`: `API_KEY`, `BASE_URL`, `MODEL_NAME`, при необходимости `MODEL_PROXY_URL`/`MODEL_CA_FILE`.
Не печатайте Settings, ключи или URL с паролем. Агентному серверу необходим tool calling.

Следующая ячейка выключена. Если осознанно включить `RUN_LIVE`, запрос уйдёт выбранному провайдеру и может быть платным;
инструменты останутся на локальной учебной SQLite. Для настоящего PostgreSQL используйте CLI основного workshop с `PostgresSandbox`.


In [13]:
RUN_LIVE = False
if RUN_LIVE:
    from harness_lab.settings import Settings
    from harness_lab.model import ModelClient
    live_settings = Settings.load(ROOT / ".env")
    local_db = SQLiteSandbox(ROWS)
    try:
        live = HarnessAgent(ModelClient(live_settings), ToolExecutor(local_db)).run(train[0]["case"], schema_first)
        print("Ответ реальной модели:", live.output)
        print("Вызовы LLM:", live.llm_calls, "ошибка:", live.error)
    finally:
        local_db.close()
else:
    print("RUN_LIVE=False: сеть и .env не использованы.")


RUN_LIVE=False: сеть и .env не использованы.


## Самостоятельная работа

1. Измените `max_steps` на 1, 2 и 3. Критерий: объяснить число policy/tool calls и наличие результата SELECT до final.
2. Сломайте одну ветвь `ScriptedPolicy`: уберите фильтр валюты и после describe. Критерий: grader видит провал, и увеличение бюджета его не скрывает.
3. Добавьте пустую популяцию или новый статус и валюту. Критерий: ожидаемая сумма определена заранее, метрика и trace объясняют результат; test не используется для следующего patch.

Не подменяйте оптимизацию весов поиском конфигурации. Не считайте наличие planning/review гарантией качества.
Программная policy и два train-примера позволяют увидеть цикл, но не позволяют выбрать лучший harness для реальной LLM.
За реальным поиском, траекториями и открытой eval-оценкой переходите к основному [README](../README.md).
